+ this notebook generates Backend-required files to view a project with the DataDiVR (preview or VR)
+ STEP 1 and the "create a graph" section contains a template graph writing a required format (json) to then use the generate-project functions of the DataDiVR backend

+ STEP 2 to actually generate BACKEND project files.

In [43]:
import networkx as nx
import json 
import os
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

# these are the two functions one needs to create a JSON file to upload and create the project in the backend 
import nx2json as nx2j
import uploaderGraph as uG

In [44]:
import numpy as np 
import random 


def generate_peripheral_position():
    coords = []
    for _ in range(3):
        edge_val = np.random.choice([0.0, 1.0])
        jitter = np.random.uniform(-0.1, 0.1)
        coords.append(np.clip(edge_val + jitter, 0.0, 1.0))
    return tuple(coords)


def generate_random_spherical_position(spatial_range=(0.8, 1.0)):
        radius = random.uniform(*spatial_range)
        theta = random.uniform(0, 2 * np.pi)
        phi = random.uniform(0, np.pi)
        x = radius * np.sin(phi) * np.cos(theta)
        y = radius * np.sin(phi) * np.sin(theta)
        z = radius * np.cos(phi)
        
        # Normalize to 0-1 range
        x = (x + 1) / 2
        y = (y + 1) / 2
        z = (z + 1) / 2
        
        return (x, y, z)


## How this is meant to be used:
+ Create an nx.Graph Object 

+ set attributes in the nx.Graph (optional, all can be empty) e.g. node positions ("pos") and colors ("nodecolor") and link colors ("linkcolor")

+ use the "create_project" function (down below to generate your project for the platform)

# make VR project 

In [45]:
# from PPI edgelist get links for each compartment based on the nodes found in the compartment files

# get links from DATA (chloe) source 
df_links_uniprot = pd.read_csv("temp-files/students/_consensus_ppi_bioplex_biogrid_intact_huri_edgelist.tsv", sep="\t", header=None)
df_links_uniprot

,0,1
0,Q8NI60,Q96M61
1,P25786,P48556
2,P05067,Q96BI3
3,P62899,P84098
4,P00492,Q14164
...,...,...
313662,P0CG48,P23246
313663,O75528,Q8N2W9
313664,P84103,Q8TBF4
313665,Q14766,Q8IXL6


In [46]:
G_uniprot = nx.from_pandas_edgelist(df_links_uniprot, source=0, target=1)
print("number of nodes in uniprot graph:", G_uniprot.number_of_nodes())
print("number of edges in uniprot graph:", G_uniprot.number_of_edges())

number of nodes in uniprot graph: 19554
number of edges in uniprot graph: 313667


In [47]:
# make edge list from df
edge_list_uniprot = list(zip(df_links_uniprot[0], df_links_uniprot[1]))
print("number of edges in the original PPI network: ", len(edge_list_uniprot))

print(edge_list_uniprot[0:5])

number of edges in the original PPI network:  313667
[('Q8NI60', 'Q96M61'), ('P25786', 'P48556'), ('P05067', 'Q96BI3'), ('P62899', 'P84098'), ('P00492', 'Q14164')]


In [48]:
# make graphs for each compartment

df_actin = pd.read_csv("temp-files/students/actinfilaments.tsv", sep="\t")
df_actin_links = df_links_uniprot[(df_links_uniprot[0].isin(df_actin['Node ID'])) & (df_links_uniprot[1].isin(df_actin['Node ID']))]
print("number of links in actin filaments: ", len(df_actin_links))

df_centro = pd.read_csv("temp-files/students/centrosome.tsv", sep="\t")
df_centro_links = df_links_uniprot[(df_links_uniprot[0].isin(df_centro['Node ID'])) & (df_links_uniprot[1].isin(df_centro['Node ID']))]
print("number of links in centrosome: ", len(df_centro_links))

df_cytosol = pd.read_csv("temp-files/students/cytosol.tsv", sep="\t")
df_cytosol_links = df_links_uniprot[(df_links_uniprot[0].isin(df_cytosol['Node ID'])) & (df_links_uniprot[1].isin(df_cytosol['Node ID']))]
print("number of links in cytosol: ", len(df_cytosol_links))

df_endoplas = pd.read_csv("temp-files/students/endoplasmicreticulum.tsv", sep="\t")
df_endoplas_links = df_links_uniprot[(df_links_uniprot[0].isin(df_endoplas['Node ID'])) & (df_links_uniprot[1].isin(df_endoplas['Node ID']))]
print("number of links in endoplasmic reticulum: ", len(df_endoplas_links))

df_golgi = pd.read_csv("temp-files/students/golgiapparatus.tsv", sep="\t")
df_golgi_links = df_links_uniprot[(df_links_uniprot[0].isin(df_golgi['Node ID'])) & (df_links_uniprot[1].isin(df_golgi['Node ID']))]
print("number of links in golgi apparatus: ", len(df_golgi_links))

df_intfil = pd.read_csv("temp-files/students/intermediatefilaments.tsv", sep="\t")
df_intfil_links = df_links_uniprot[(df_links_uniprot[0].isin(df_intfil['Node ID'])) & (df_links_uniprot[1].isin(df_intfil['Node ID']))]
print("number of links in intermediate filaments: ", len(df_intfil_links))

df_microtub = pd.read_csv("temp-files/students/microtubules.tsv", sep="\t")
df_microtub_links = df_links_uniprot[(df_links_uniprot[0].isin(df_microtub['Node ID'])) & (df_links_uniprot[1].isin(df_microtub['Node ID']))]
print("number of links in microtubules: ", len(df_microtub_links))

df_mito = pd.read_csv("temp-files/students/mitochondria.tsv", sep="\t")
df_mito_links = df_links_uniprot[(df_links_uniprot[0].isin(df_mito['Node ID'])) & (df_links_uniprot[1].isin(df_mito['Node ID']))]
print("number of links in mitochondria: ", len(df_mito_links))

df_nuclearmem = pd.read_csv("temp-files/students/nuclearmembrane.tsv", sep="\t")
df_nuclearmem_links = df_links_uniprot[(df_links_uniprot[0].isin(df_nuclearmem['Node ID'])) & (df_links_uniprot[1].isin(df_nuclearmem['Node ID']))]
print("number of links in nuclear membrane: ", len(df_nuclearmem_links))

df_nucleoli = pd.read_csv("temp-files/students/nucleoli.tsv", sep="\t")
df_nucleoli_links = df_links_uniprot[(df_links_uniprot[0].isin(df_nucleoli['Node ID'])) & (df_links_uniprot[1].isin(df_nucleoli['Node ID']))]
print("number of links in nucleoli: ", len(df_nucleoli_links))

df_nuclplasm = pd.read_csv("temp-files/students/nucleoplasm.tsv", sep="\t")
df_nuclplasm_links = df_links_uniprot[(df_links_uniprot[0].isin(df_nuclplasm['Node ID'])) & (df_links_uniprot[1].isin(df_nuclplasm['Node ID']))]
print("number of links in nucleoplasm: ", len(df_nuclplasm_links))

df_plasmmem = pd.read_csv("temp-files/students/plasmamembrane.tsv", sep="\t")
df_plasmmem_links = df_links_uniprot[(df_links_uniprot[0].isin(df_plasmmem['Node ID'])) & (df_links_uniprot[1].isin(df_plasmmem['Node ID']))]
print("number of links in plasma membrane: ", len(df_plasmmem_links))

df_prim = pd.read_csv("temp-files/students/primarycilium.tsv", sep="\t")
df_prim_links = df_links_uniprot[(df_links_uniprot[0].isin(df_prim['Node ID'])) & (df_links_uniprot[1].isin(df_prim['Node ID']))]
print("number of links in primary cilium: ", len(df_prim_links))

number of links in actin filaments:  2907
number of links in centrosome:  2005
number of links in cytosol:  74909
number of links in endoplasmic reticulum:  6439
number of links in golgi apparatus:  3558
number of links in intermediate filaments:  200
number of links in microtubules:  2392
number of links in mitochondria:  3664
number of links in nuclear membrane:  202
number of links in nucleoli:  6994
number of links in nucleoplasm:  78035
number of links in plasma membrane:  27873
number of links in primary cilium:  1407


In [49]:
# annotation dict 
# include IDs (gene sym and uniprot),compartment and source 

df_n_comp = pd.read_csv("temp-files/students/_protein_location_HPA_GO.tsv", sep="\t")
df_uniprot_genesym = pd.read_csv("temp-files/students/_d_uniprot_to_symbol_20251120.tsv", sep="\t")
df_uniprot_genesym_dict = pd.Series(df_uniprot_genesym.symbol.values,index=df_uniprot_genesym.uniprotid).to_dict()

# add compartment annotation to each node dict
d_annot_actin = {
    row['Node ID']: {
        'compartment': 'actin filaments',
        'gene symbol': df_uniprot_genesym_dict.get(row['Node ID'], 'unknown'),
        'uniprot' : row['Node ID'],
        'community_assigned: ' : row['community_id'],
        'source': df_n_comp[df_n_comp['protein'] == df_uniprot_genesym_dict.get(row['Node ID'], 'unknown')]['source(s)'].values[0]
        if df_uniprot_genesym_dict.get(row['Node ID'], 'unknown') in df_n_comp['protein'].values else 'unknown'
    }
    for index, row in df_actin.iterrows()
}
d_annot_centro = {
    row['Node ID']: {
        'compartment': 'centrosome',
        'gene symbol': df_uniprot_genesym_dict.get(row['Node ID'], 'unknown'),
        'uniprot' : row['Node ID'],
        'community_assigned: ' : row['community_id'],
        'source': df_n_comp[df_n_comp['protein'] == df_uniprot_genesym_dict.get(row['Node ID'], 'unknown')]['source(s)'].values[0]
        if df_uniprot_genesym_dict.get(row['Node ID'], 'unknown') in df_n_comp['protein'].values else 'unknown'
    }
    for index, row in df_centro.iterrows()
}
d_annot_cytosol = {
    row['Node ID']: {
        'compartment': 'cytosol',
        'gene symbol': df_uniprot_genesym_dict.get(row['Node ID'], 'unknown'),
        'uniprot': row['Node ID'],
        'community_assigned: ' : row['community_id'],
        'source': df_n_comp[df_n_comp['protein'] == df_uniprot_genesym_dict.get(row['Node ID'], 'unknown')]['source(s)'].values[0]
        if df_uniprot_genesym_dict.get(row['Node ID'], 'unknown') in df_n_comp['protein'].values else 'unknown'
    }   
    for index, row in df_cytosol.iterrows()
}
d_annot_endoplas = {
    row['Node ID']: {
        'compartment': 'endoplasmic reticulum',
        'gene symbol': df_uniprot_genesym_dict.get(row['Node ID'], 'unknown'),
        'uniprot': row['Node ID'],
        'community_assigned: ' : row['community_id'],
        'source': df_n_comp[df_n_comp['protein'] == df_uniprot_genesym_dict.get(row['Node ID'], 'unknown')]['source(s)'].values[0]
        if df_uniprot_genesym_dict.get(row['Node ID'], 'unknown') in df_n_comp['protein'].values else 'unknown'
    }   
    for index, row in df_endoplas.iterrows()
}
d_annot_golgi = {
    row['Node ID']: {
        'compartment': 'golgi apparatus',
        'gene symbol': df_uniprot_genesym_dict.get(row['Node ID'], 'unknown'),
        'uniprot': row['Node ID'],
        'community_assigned: ' : row['community_id'],
        'source': df_n_comp[df_n_comp['protein'] == df_uniprot_genesym_dict.get(row['Node ID'], 'unknown')]['source(s)'].values[0]
        if df_uniprot_genesym_dict.get(row['Node ID'], 'unknown') in df_n_comp['protein'].values else 'unknown'
    }
    for index, row in df_golgi.iterrows()
}   
d_annot_intfil = {
    row['Node ID']: {
        'compartment': 'intermediate filaments',
        'gene symbol': df_uniprot_genesym_dict.get(row['Node ID'], 'unknown'),
        'uniprot': row['Node ID'],
        'community_assigned: ' : row['community_id'],
        'source': df_n_comp[df_n_comp['protein'] == df_uniprot_genesym_dict.get(row['Node ID'], 'unknown')]['source(s)'].values[0]
        if df_uniprot_genesym_dict.get(row['Node ID'], 'unknown') in df_n_comp['protein'].values else 'unknown'
    }   
    for index, row in df_intfil.iterrows()
}
d_annot_microtub = {
    row['Node ID']: {
        'compartment': 'microtubules',
        'gene symbol': df_uniprot_genesym_dict.get(row['Node ID'], 'unknown'),
        'uniprot': row['Node ID'],
        'community_assigned: ' : row['community_id'],
        'source': df_n_comp[df_n_comp['protein'] == df_uniprot_genesym_dict.get(row['Node ID'], 'unknown')]['source(s)'].values[0]
        if df_uniprot_genesym_dict.get(row['Node ID'], 'unknown') in df_n_comp['protein'].values else 'unknown'
    }   
    for index, row in df_microtub.iterrows()
}
d_annot_mito = {
    row['Node ID']: {
        'compartment': 'mitochondria',
        'gene symbol': df_uniprot_genesym_dict.get(row['Node ID'], 'unknown'),
        'uniprot': row['Node ID'],
        'community_assigned: ' : row['community_id'],
        'source': df_n_comp[df_n_comp['protein'] == df_uniprot_genesym_dict.get(row['Node ID'], 'unknown')]['source(s)'].values[0]
        if df_uniprot_genesym_dict.get(row['Node ID'], 'unknown') in df_n_comp['protein'].values else 'unknown'
    }   
    for index, row in df_mito.iterrows()
}
d_annot_nuclearmem = {
    row['Node ID']: {
        'compartment': 'nuclear membrane',
        'gene symbol': df_uniprot_genesym_dict.get(row['Node ID'], 'unknown'),
        'uniprot': row['Node ID'],
        'community_assigned: ' : row['community_id'],
        'source': df_n_comp[df_n_comp['protein'] == df_uniprot_genesym_dict.get(row['Node ID'], 'unknown')]['source(s)'].values[0]
        if df_uniprot_genesym_dict.get(row['Node ID'], 'unknown') in df_n_comp['protein'].values else 'unknown'
    }   
    for index, row in df_nuclearmem.iterrows()
}
d_annot_nucleoli = {
    row['Node ID']: {
        'compartment': 'nucleoli',
        'gene symbol': df_uniprot_genesym_dict.get(row['Node ID'], 'unknown'),
        'uniprot': row['Node ID'],
        'community_assigned: ' : row['community_id'],
        'source': df_n_comp[df_n_comp['protein'] == df_uniprot_genesym_dict.get(row['Node ID'], 'unknown')]['source(s)'].values[0]
        if df_uniprot_genesym_dict.get(row['Node ID'], 'unknown') in df_n_comp['protein'].values else 'unknown'
    }   
    for index, row in df_nucleoli.iterrows()
}
d_annot_nuclplasm = {
    row['Node ID']: {
        'compartment': 'nucleoplasm',
        'gene symbol': df_uniprot_genesym_dict.get(row['Node ID'], 'unknown'),
        'uniprot': row['Node ID'],
        'community_assigned: ' : row['community_id'],
        'source': df_n_comp[df_n_comp['protein'] == df_uniprot_genesym_dict.get(row['Node ID'], 'unknown')]['source(s)'].values[0]
        if df_uniprot_genesym_dict.get(row['Node ID'], 'unknown') in df_n_comp['protein'].values else 'unknown'
    }   
    for index, row in df_nuclplasm.iterrows()
}
d_annot_plasmmem = {
    row['Node ID']: {
        'compartment': 'plasma membrane',
        'gene symbol': df_uniprot_genesym_dict.get(row['Node ID'], 'unknown'),
        'uniprot': row['Node ID'],
        'community_assigned: ' : row['community_id'],
        'source': df_n_comp[df_n_comp['protein'] == df_uniprot_genesym_dict.get(row['Node ID'], 'unknown')]['source(s)'].values[0]
        if df_uniprot_genesym_dict.get(row['Node ID'], 'unknown') in df_n_comp['protein'].values else 'unknown'
    }   
    for index, row in df_plasmmem.iterrows()
}
d_annot_prim = {
    row['Node ID']: {
        'compartment': 'primary cilium',
        'gene symbol': df_uniprot_genesym_dict.get(row['Node ID'], 'unknown'),
        'uniprot': row['Node ID'],
        'community_assigned: ' : row['community_id'],
        'source': df_n_comp[df_n_comp['protein'] == df_uniprot_genesym_dict.get(row['Node ID'], 'unknown')]['source(s)'].values[0]
        if df_uniprot_genesym_dict.get(row['Node ID'], 'unknown') in df_n_comp['protein'].values else 'unknown'
    }   
    for index, row in df_prim.iterrows()
}

### make nx Graph objects for each compartment 

In [50]:
G_actin = nx.from_pandas_edgelist(df_actin_links, 0, 1)
G_actin.add_nodes_from(df_actin['Node ID'].tolist())  # Ensure all nodes from df_actin are included
d_actin_pos = dict(zip(G_actin.nodes(),list(zip(df_actin['x'], df_actin['y'], df_actin['z']))))
nx.set_node_attributes(G_actin, d_actin_pos, 'pos')

d_actin_col = dict(zip(G_actin.nodes(),list(zip(df_actin['r'], df_actin['g'], df_actin['b'], df_actin['a']))))
nx.set_node_attributes(G_actin, d_actin_col, 'nodecolor')

nx.set_node_attributes(G_actin, d_annot_actin, 'annotation')

# remove nodes from G_actin in G_uniprot - THIS NEEDS TO BE DONE BEFORE RELABELLING
#G_uniprot.remove_nodes_from(G_actin.nodes())

# relabel nodes to include short ID for compartment
mapping_actin = {node: f"actin_{node}" for node in G_actin.nodes()}
G_actin = nx.relabel_nodes(G_actin, mapping_actin)

print("number of nodes in actin graph: ", len(G_actin.nodes()))
print("number of edges in actin graph: ", len(G_actin.edges()))



G_centro = nx.from_pandas_edgelist(df_centro_links, 0, 1)
G_centro.add_nodes_from(df_centro['Node ID'].tolist())  # Ensure all nodes from df_centro are included
d_centro_pos = dict(zip(G_centro.nodes(), list(zip(df_centro['x'], df_centro['y'], df_centro['z']))))
nx.set_node_attributes(G_centro, d_centro_pos, 'pos')

d_centro_col = dict(zip(G_centro.nodes(), list(zip(df_centro['r'], df_centro['g'], df_centro['b'], df_centro['a']))))
nx.set_node_attributes(G_centro, d_centro_col, 'nodecolor')

nx.set_node_attributes(G_centro, d_annot_centro, 'annotation')

# remove nodes in G_uniprot
#G_uniprot.remove_nodes_from(G_centro.nodes())

# relabel nodes to include short ID for compartment
mapping_centro = {node: f"centro_{node}" for node in G_centro.nodes()}
G_centro = nx.relabel_nodes(G_centro, mapping_centro)

print("number of nodes in centrosome graph: ", len(G_centro.nodes()))
print("number of edges in centrosome graph: ", len(G_centro.edges()))



G_cytosol = nx.from_pandas_edgelist(df_cytosol_links, 0, 1)
G_cytosol.add_nodes_from(df_cytosol['Node ID'].tolist())  # Ensure all nodes from df_cytosol are included
d_cytosol_pos = dict(zip(G_cytosol.nodes(), list(zip(df_cytosol['x'], df_cytosol['y'], df_cytosol['z']))))
nx.set_node_attributes(G_cytosol, d_cytosol_pos, 'pos')

d_cytosol_col = dict(zip(G_cytosol.nodes(), list(zip(df_cytosol['r'], df_cytosol['g'], df_cytosol['b'], df_cytosol['a'])))) 
nx.set_node_attributes(G_cytosol, d_cytosol_col, 'nodecolor')

nx.set_node_attributes(G_cytosol, d_annot_cytosol, 'annotation')

# remove nodes in G_uniprot
#G_uniprot.remove_nodes_from(G_cytosol.nodes())

# relabel nodes to include short ID for compartment
mapping_cytosol = {node: f"cytosol_{node}" for node in G_cytosol.nodes()}
G_cytosol = nx.relabel_nodes(G_cytosol, mapping_cytosol)

print("number of nodes in cytosol graph: ", len(G_cytosol.nodes()))
print("number of edges in cytosol graph: ", len(G_cytosol.edges()))



G_endoplas = nx.from_pandas_edgelist(df_endoplas_links, 0, 1)
G_endoplas.add_nodes_from(df_endoplas['Node ID'].tolist())  # Ensure all nodes from df_endoplas are included
d_endoplas_pos = dict(zip(G_endoplas.nodes(), list(zip(df_endoplas['x'], df_endoplas['y'], df_endoplas['z']))))
nx.set_node_attributes(G_endoplas, d_endoplas_pos, 'pos')

d_endoplas_col = dict(zip(G_endoplas.nodes(), list(zip(df_endoplas['r'], df_endoplas['g'], df_endoplas['b'], df_endoplas['a']))))
nx.set_node_attributes(G_endoplas, d_endoplas_col, 'nodecolor')

nx.set_node_attributes(G_endoplas, d_annot_endoplas, 'annotation')

# remove nodes in G_uniprot
#G_uniprot.remove_nodes_from(G_endoplas.nodes())

# relabel nodes to include short ID for compartment
mapping_endoplas = {node: f"endoplas_{node}" for node in G_endoplas.nodes()}
G_endoplas = nx.relabel_nodes(G_endoplas, mapping_endoplas)

print("number of nodes in endoplasmic reticulum graph: ", len(G_endoplas.nodes()))
print("number of edges in endoplasmic reticulum graph: ", len(G_endoplas.edges()))



G_golgi = nx.from_pandas_edgelist(df_golgi_links, 0, 1)
G_golgi.add_nodes_from(df_golgi['Node ID'].tolist())  # Ensure all nodes from df_golgi are included
d_golgi_pos = dict(zip(G_golgi.nodes(), list(zip(df_golgi['x'], df_golgi['y'], df_golgi['z']))))
nx.set_node_attributes(G_golgi, d_golgi_pos, 'pos')

d_golgi_col = dict(zip(G_golgi.nodes(), list(zip(df_golgi['r'], df_golgi['g'], df_golgi['b'], df_golgi['a']))))
nx.set_node_attributes(G_golgi, d_golgi_col, 'nodecolor')

nx.set_node_attributes(G_golgi, d_annot_golgi, 'annotation')

# remove nodes in G_uniprot
#G_uniprot.remove_nodes_from(G_golgi.nodes())

# relabel nodes to include short ID for compartment
mapping_golgi = {node: f"golgi_{node}" for node in G_golgi.nodes()}
G_golgi = nx.relabel_nodes(G_golgi, mapping_golgi)

print("number of nodes in golgi graph: ", len(G_golgi.nodes()))
print("number of edges in golgi graph: ", len(G_golgi.edges()))



G_intfil = nx.from_pandas_edgelist(df_intfil_links, 0, 1)
G_intfil.add_nodes_from(df_intfil['Node ID'].tolist())  # Ensure all nodes from df_intfil are included
d_intfil_pos = dict(zip(G_intfil.nodes(), list(zip(df_intfil['x'], df_intfil['y'], df_intfil['z']))))
nx.set_node_attributes(G_intfil, d_intfil_pos, 'pos')

d_intfil_col = dict(zip(G_intfil.nodes(), list(zip(df_intfil['r'], df_intfil['g'], df_intfil['b'], df_intfil['a']))))
nx.set_node_attributes(G_intfil, d_intfil_col, 'nodecolor')

nx.set_node_attributes(G_intfil, d_annot_intfil, 'annotation')

# remove nodes in G_uniprot
#G_uniprot.remove_nodes_from(G_intfil.nodes())

# relabel nodes to include short ID for compartment
mapping_intfil = {node: f"intfil_{node}" for node in G_intfil.nodes()}
G_intfil = nx.relabel_nodes(G_intfil, mapping_intfil)

print("number of nodes in intermediate filaments graph: ", len(G_intfil.nodes()))
print("number of edges in intermediate filaments graph: ", len(G_intfil.edges()))



G_microtub = nx.from_pandas_edgelist(df_microtub_links, 0, 1)
G_microtub.add_nodes_from(df_microtub['Node ID'].tolist())  # Ensure all nodes from df_microtub are included
d_microtub_pos = dict(zip(G_microtub.nodes(), list(zip(df_microtub['x'], df_microtub['y'], df_microtub['z']))))
nx.set_node_attributes(G_microtub, d_microtub_pos, 'pos')

d_microtub_col = dict(zip(G_microtub.nodes(), list(zip(df_microtub['r'], df_microtub['g'], df_microtub['b'], df_microtub['a']))))
nx.set_node_attributes(G_microtub, d_microtub_col, 'nodecolor')

nx.set_node_attributes(G_microtub, d_annot_microtub, 'annotation')

# remove nodes in G_uniprot
#G_uniprot.remove_nodes_from(G_microtub.nodes())

# relabel nodes to include short ID for compartment
mapping_microtub = {node: f"microtub_{node}" for node in G_microtub.nodes()}
G_microtub = nx.relabel_nodes(G_microtub, mapping_microtub)

print("number of nodes in microtubules graph: ", len(G_microtub.nodes()))
print("number of edges in microtubules graph: ", len(G_microtub.edges()))



G_mito = nx.from_pandas_edgelist(df_mito_links, 0,  1)
G_mito.add_nodes_from(df_mito['Node ID'].tolist())  # Ensure all nodes from df_mito are included
d_mito = dict(zip(G_mito.nodes(), list(zip(df_mito['x'], df_mito['y'], df_mito['z']))))
nx.set_node_attributes(G_mito, d_mito, 'pos')

d_mito_col = dict(zip(G_mito.nodes(), list(zip(df_mito['r'], df_mito['g'], df_mito['b'], df_mito['a']))))
nx.set_node_attributes(G_mito, d_mito_col, 'nodecolor')

nx.set_node_attributes(G_mito, d_annot_mito, 'annotation')

# remove nodes in G_uniprot
#G_uniprot.remove_nodes_from(G_mito.nodes())

# relabel nodes to include short ID for compartment
mapping_mito = {node: f"mito_{node}" for node in G_mito.nodes()}
G_mito = nx.relabel_nodes(G_mito, mapping_mito)

print("number of nodes in mitochondria graph: ", len(G_mito.nodes()))
print("number of edges in mitochondria graph: ", len(G_mito.edges()))



G_nuclearmem = nx.from_pandas_edgelist(df_nuclearmem_links, 0, 1)
G_nuclearmem.add_nodes_from(df_nuclearmem['Node ID'].tolist())  # Ensure all nodes from df_nuclearmem are included
d_nuclearmem = dict(zip(G_nuclearmem.nodes(), list(zip(df_nuclearmem['x'], df_nuclearmem['y'], df_nuclearmem['z']))))
nx.set_node_attributes(G_nuclearmem, d_nuclearmem, 'pos')

d_nuclearmem_col = dict(zip(G_nuclearmem.nodes(), list(zip(df_nuclearmem['r'], df_nuclearmem['g'], df_nuclearmem['b'], df_nuclearmem['a']))))
nx.set_node_attributes(G_nuclearmem, d_nuclearmem_col, 'nodecolor')

nx.set_node_attributes(G_nuclearmem, d_annot_nuclearmem, 'annotation')

# remove nodes in G_uniprot
#G_uniprot.remove_nodes_from(G_nuclearmem.nodes())

# relabel nodes to include short ID for compartment
mapping_nuclearmem = {node: f"nuclearmem_{node}" for node in G_nuclearmem.nodes()}
G_nuclearmem = nx.relabel_nodes(G_nuclearmem, mapping_nuclearmem)

print("number of nodes in nuclear membrane graph: ", len(G_nuclearmem.nodes()))
print("number of edges in nuclear membrane graph: ", len(G_nuclearmem.edges()))



G_nucleoli = nx.from_pandas_edgelist(df_nucleoli_links, 0, 1) 
G_nucleoli.add_nodes_from(df_nucleoli['Node ID'].tolist())  # Ensure all nodes from df_nucleoli are included
d_nucleoli = dict(zip(G_nucleoli.nodes(), list(zip(df_nucleoli['x'], df_nucleoli['y'], df_nucleoli['z']))))
nx.set_node_attributes(G_nucleoli, d_nucleoli, 'pos')  

d_nucleoli_col = dict(zip(G_nucleoli.nodes(), list(zip(df_nucleoli['r'], df_nucleoli['g'], df_nucleoli['b'], df_nucleoli['a']))))
nx.set_node_attributes(G_nucleoli, d_nucleoli_col, 'nodecolor')

nx.set_node_attributes(G_nucleoli, d_annot_nucleoli, 'annotation')

# remove nodes in G_uniprot
#G_uniprot.remove_nodes_from(G_nucleoli.nodes())

# relabel nodes to include short ID for compartment
mapping_nucleoli = {node: f"nucleoli_{node}" for node in G_nucleoli.nodes()}
G_nucleoli = nx.relabel_nodes(G_nucleoli, mapping_nucleoli)

print("number of nodes in nucleoli graph: ", len(G_nucleoli.nodes()))
print("number of edges in nucleoli graph: ", len(G_nucleoli.edges()))



G_nuclplasm = nx.from_pandas_edgelist(df_nuclplasm_links, 0, 1)
G_nuclplasm.add_nodes_from(df_nuclplasm['Node ID'].tolist())  # Ensure all nodes from df_nuclplasm are included
d_nuclplasm_pos = dict(zip(G_nuclplasm.nodes(), list(zip(df_nuclplasm['x'], df_nuclplasm['y'], df_nuclplasm['z']))))
nx.set_node_attributes(G_nuclplasm, d_nuclplasm_pos, 'pos')

d_nuclplasm_col = dict(zip(G_nuclplasm.nodes(), list(zip(df_nuclplasm['r'], df_nuclplasm['g'], df_nuclplasm['b'], df_nuclplasm['a']))))
nx.set_node_attributes(G_nuclplasm, d_nuclplasm_col, 'nodecolor')

nx.set_node_attributes(G_nuclplasm, d_annot_nuclplasm, 'annotation')

# remove nodes in G_uniprot
#G_uniprot.remove_nodes_from(G_nuclplasm.nodes())

# relabel  nodes to include short ID for compartment
mapping_nuclplasm = {node: f"nuclplasm_{node}" for node in G_nuclplasm.nodes()}
G_nuclplasm = nx.relabel_nodes(G_nuclplasm, mapping_nuclplasm)

print("number of nodes in nucleoplasm graph: ", len(G_nuclplasm.nodes()))
print("number of edges in nucleoplasm graph: ", len(G_nuclplasm.edges()))



G_plasmmem = nx.from_pandas_edgelist(df_plasmmem_links, 0, 1)
G_plasmmem.add_nodes_from(df_plasmmem['Node ID'].tolist())  # Ensure all nodes from df_plasmmem are included
d_plasmmem_pos = dict(zip(G_plasmmem.nodes(), list(zip(df_plasmmem['x'], df_plasmmem['y'], df_plasmmem['z']))))
nx.set_node_attributes(G_plasmmem, d_plasmmem_pos, 'pos')

d_plasmmem_col = dict(zip(G_plasmmem.nodes(), list(zip(df_plasmmem['r'], df_plasmmem['g'], df_plasmmem['b'], df_plasmmem['a']))))
nx.set_node_attributes(G_plasmmem, d_plasmmem_col, 'nodecolor')

nx.set_node_attributes(G_plasmmem, d_annot_plasmmem, 'annotation')

# remove nodes in G_uniprot
#G_uniprot.remove_nodes_from(G_plasmmem.nodes())

# relabel nodes to include short ID for compartment
mapping_plasmmem = {node: f"plasmmem_{node}" for node in G_plasmmem.nodes()}
G_plasmmem = nx.relabel_nodes(G_plasmmem, mapping_plasmmem)

print("number of nodes in plasma membrane graph: ", len(G_plasmmem.nodes()))
print("number of edges in plasma membrane graph: ", len(G_plasmmem.edges()))



G_prim = nx.from_pandas_edgelist(df_prim_links, 0, 1)
G_prim.add_nodes_from(df_prim['Node ID'].tolist())  # Ensure all nodes from df_prim are included
d_prim_pos = dict(zip(G_prim.nodes(),list(zip(df_prim['x'], df_prim['y'], df_prim['z']))))
nx.set_node_attributes(G_prim, d_prim_pos, 'pos')  

d_prim_col = dict(zip(G_prim.nodes(), list(zip(df_prim['r'], df_prim['g'], df_prim['b'], df_prim['a']))))
nx.set_node_attributes(G_prim, d_prim_col, 'nodecolor')

nx.set_node_attributes(G_prim, d_annot_prim, 'annotation')

# remove nodes in G_uniprot
#G_uniprot.remove_nodes_from(G_prim.nodes())

# relabel nodes to include short ID for compartment
mapping_prim = {node: f"prim_{node}" for node in G_prim.nodes()}
G_prim = nx.relabel_nodes(G_prim, mapping_prim)

print("number of nodes in primary cilium graph: ", len(G_prim.nodes()))
print("number of edges in primary cilium graph: ", len(G_prim.edges()))

number of nodes in actin graph:  492
number of edges in actin graph:  2907
number of nodes in centrosome graph:  493
number of edges in centrosome graph:  2005
number of nodes in cytosol graph:  5738
number of edges in cytosol graph:  74909
number of nodes in endoplasmic reticulum graph:  1146
number of edges in endoplasmic reticulum graph:  6439
number of nodes in golgi graph:  1009
number of edges in golgi graph:  3558
number of nodes in intermediate filaments graph:  54
number of edges in intermediate filaments graph:  200
number of nodes in microtubules graph:  576
number of edges in microtubules graph:  2392
number of nodes in mitochondria graph:  364
number of edges in mitochondria graph:  3664
number of nodes in nuclear membrane graph:  88
number of edges in nuclear membrane graph:  202
number of nodes in nucleoli graph:  881
number of edges in nucleoli graph:  6994
number of nodes in nucleoplasm graph:  5555
number of edges in nucleoplasm graph:  78035
number of nodes in plasma

In [51]:
# all into one graphlist to make project in VR 

# Create a new graph object to include all data
G = nx.Graph()

# Add nodes and edges from each compartment graph
for graph in [G_actin, G_centro, G_cytosol, G_endoplas, G_golgi, G_intfil, G_microtub, G_mito, G_nuclearmem, G_nucleoli, G_nuclplasm, G_plasmmem, G_prim,
              # consider adding : G_uniprot
              ]:
    G.add_nodes_from(graph.nodes(data=True))
    G.add_edges_from(graph.edges(data=True))




# # ADD IN FUTURE: e.g. for Chloé's Data Story 
# # add edges from uniprot-edge-list
# G.add_edges_from(edge_list_uniprot)

# # replace node labels if ID matches with number in individual graphs e.g. actin_P12345 -> P12345
# mapping_uniprot = {}
# for node in G_uniprot.nodes():
#     for g in [G_actin, G_centro, G_cytosol, G_endoplas, G_golgi, G_intfil, G_microtub, G_mito, G_nuclearmem, G_nucleoli, G_nuclplasm, G_plasmmem, G_prim]:
#         if node in g.nodes():
#             mapping_uniprot[node] = node
#             break
# G = nx.relabel_nodes(G, mapping_uniprot)

# # remove nodes which are not in any compartment graphs
# nodes_to_remove = [node for node in G_uniprot.nodes() if node not in mapping_uniprot]
# G.remove_nodes_from(nodes_to_remove)

# # TO FIX ??? 
# # # fix pos and nodecolor missing for these edges / nodes
# #         if 'pos' not in G.nodes[edge[0]]:
# #             G.nodes[edge[0]]['pos'] = generate_random_spherical_position((0.45,0.55))
# #             G.nodes[edge[0]]['nodecolor'] = (255,0,0,200) # DEBUG # grey color for unknown location
# #         if 'pos' not in G.nodes[edge[1]]:
# #             G.nodes[edge[1]]['pos'] = generate_random_spherical_position((0.45,0.55))
# #             G.nodes[edge[1]]['nodecolor'] = (255,0,0,200) # DEBUG # grey color for unknown location




print("Number of nodes in the combined graph: ", len(G.nodes()))
print("Number of edges in the combined graph: ", len(G.edges()))

Number of nodes in the combined graph:  20324
Number of edges in the combined graph:  210585


In [52]:
# make lookup for node of G.nodes and 
node_lookup = {}
for n in G.nodes():
    short_id = n.split("_",1)[1]  # remove compartment prefix
    node_lookup[n] = short_id
node_lookup

{'actin_P62899': 'P62899',
 'actin_P84098': 'P84098',
 'actin_P35221': 'P35221',
 'actin_Q9UGI8': 'Q9UGI8',
 'actin_O15296': 'O15296',
 'actin_Q96AC1': 'Q96AC1',
 'actin_O43707': 'O43707',
 'actin_P00533': 'P00533',
 'actin_P15880': 'P15880',
 'actin_P18124': 'P18124',
 'actin_Q8WYJ6': 'Q8WYJ6',
 'actin_Q92599': 'Q92599',
 'actin_P05387': 'P05387',
 'actin_P62829': 'P62829',
 'actin_Q14940': 'Q14940',
 'actin_Q99653': 'Q99653',
 'actin_P53355': 'P53355',
 'actin_Q9ULV4': 'Q9ULV4',
 'actin_P30050': 'P30050',
 'actin_P62888': 'P62888',
 'actin_P62140': 'P62140',
 'actin_Q9P0K7': 'Q9P0K7',
 'actin_P11940': 'P11940',
 'actin_P46777': 'P46777',
 'actin_Q13153': 'Q13153',
 'actin_P62277': 'P62277',
 'actin_P62701': 'P62701',
 'actin_P18206': 'P18206',
 'actin_Q9Y490': 'Q9Y490',
 'actin_P62424': 'P62424',
 'actin_Q5VT25': 'Q5VT25',
 'actin_O60292': 'O60292',
 'actin_P62745': 'P62745',
 'actin_Q15599': 'Q15599',
 'actin_P62736': 'P62736',
 'actin_Q92574': 'Q92574',
 'actin_P29317': 'P29317',
 

In [53]:
# normalize all positions to range 0-1
pos = nx.get_node_attributes(G, 'pos')

x_values = [p[0] for p in pos.values()]
y_values = [p[1] for p in pos.values()]
z_values = [p[2] for p in pos.values()]

min_x, max_x = min(x_values), max(x_values)
min_y, max_y = min(y_values), max(y_values)
min_z, max_z = min(z_values), max(z_values)

for node, p in pos.items():
    norm_x = (p[0] - min_x) / (max_x - min_x) if max_x > min_x else 0.0
    norm_y = (p[1] - min_y) / (max_y - min_y) if max_y > min_y else 0.0
    norm_z = (p[2] - min_z) / (max_z - min_z) if max_z > min_z else 0.0
    G.nodes[node]['pos'] = (norm_x, norm_y, norm_z)

In [54]:
G.graph['projectname'] = "SpatialPPI_onescene"
G.graph['info'] = "A spatial PPI graph for testing purposes. Number of nodes: "+str(len(G.nodes()))+", Links: "+ str(len(G.edges()))+"."
G.graph["layoutname"] = '03_SpatialPPI'

## create VR basis project 

In [55]:
nx2j.create_project(G)

Successfully created the directory static/projects/SpatialPPI_onescene 
PROGRESS: loaded graph JSON...
PROGRESS: stored graph data...
PROGRESS: stored layouts...
PROGRESS: stored node info...
PROGRESS: made node position textures...
PROGRESS: made textures for node colors...
PROGRESS: made textures for links...
PROGRESS: no linkcolors detected for  03_SpatialPPI
PROGRESS: writing json files for project and nodes...
Project created successfully.


✅ Connected to /main


## Realtime scenes

In [24]:
import networkx as nx 
from dataXplorer import JupyterClient
from dataXplorer import AnalysisToolkit, SessionManager, TextureGenerator, ProjectFileManager, VisualizerSyncer

In [25]:
# run backend (server) using buildandrun powershell script
client = JupyterClient()

#client.disconnect()

✅ Connected to /main


✅ Connected to /main


In [26]:
# see if new project is in projectlist
import GlobalData as GD

allprojects_updated = []
for i, proj in enumerate(GD.listProjects()):
    allprojects_updated.append((i,proj))
allprojects_updated

[(0, 'AE_Memes_2022'),
 (1, 'ARS23_memes'),
 (2, 'ByzNet-1400-people-only'),
 (3, 'ByzNet_PxL'),
 (4, 'CDK5'),
 (5, 'CircLadderGraph-xsmall'),
 (6, 'diffusion'),
 (7, 'JSON_autocore'),
 (8, 'Microplastics_HumanHealth'),
 (9, 'Pesticides_HumanHealth'),
 (10, 'PG_NEW'),
 (11, 'Powergrid_Europe'),
 (12, 'PPI_brain_infarction'),
 (13, 'PPI_joel_daniel_aryan_C'),
 (14, 'PPI_networkcartoGRAPHs'),
 (15, 'SpatialPPI'),
 (16, 'SpatialPPI_'),
 (17, 'Sphere_Torus'),
 (18, 'Teapot'),
 (19, 'TheMandelbulb_edges')]

In [30]:
# select a project to work with
sel_id = 16
sel_name = allprojects_updated[sel_id][1]

# load the project data
session = SessionManager(sel_id, sel_name, client)
session.load_graph_from_project()
session.reload_project()

# initialize 
file_mgr = ProjectFileManager(session)
tex_gen = TextureGenerator(session)
syncer = VisualizerSyncer(session)

tools = AnalysisToolkit(session, tex_gen, syncer, file_mgr)

Session Graph loaded from project folder. 
Project name:  SpatialPPI_
Data: Nodes: 20324 Links: 210585


✅ Connected to /main


### scene 1 - nucleoli 

In [31]:
# get all nodes with attrlist "compartment" "nucleoli"

nodelist_nucleoli = []
for n, attr in G.nodes(data=True):
    if 'annotation' in attr and 'compartment' in attr['annotation']:
        if attr['annotation']['compartment'] == 'nucleoli':
            nodelist_nucleoli.append(n)
print("number of nodes in nucleoli compartment: ", len(nodelist_nucleoli))
print("Test: number should match the one above: ", len(G_nucleoli.nodes()))

number of nodes in nucleoli compartment:  881
Test: number should match the one above:  881


In [32]:
# get all links between nucleoli nodes
link_col_scene0 = {}
linklist = []

for i, (u, v) in enumerate(G.edges()):
    if u in nodelist_nucleoli and v in nodelist_nucleoli:
        link_col_scene0[i] = G.nodes[u]['nodecolor']  # use node color for link color
        linklist.append((u,v))
    else:
        link_col_scene0[i] = (0,0,0,0)  # transparent for non nucleoli links

# map node IDs to consecutive integers for vis
node_id_map = {node: idx for idx, node in enumerate(G.nodes())}
# Create a dictionary with original links as keys and mapped links as values
link_mapping = {(u, v): (node_id_map[u], node_id_map[v]) for u, v in G.edges()}


# adapt link_col_scene1 to new link IDs
link_col_scene0_mapped = {
    link_mapping[(u, v)]: link_col_scene0[idx]
    for idx, (u, v) in enumerate(G.edges())
}

print("number of colored / visible links in scene0: ", len(linklist))

number of colored / visible links in scene0:  6994


In [33]:
# prepare vis parameters like node colors, link colors
import random

node_col_scene1 = {}
node_pos_scene1 = {}

for i,n in enumerate(G.nodes()):
    if n in nodelist_nucleoli:
        node_col_scene1[i] = G.nodes[n]['nodecolor']
        node_pos_scene1[i] = G.nodes[n]['pos']

    elif n not in nodelist_nucleoli:
        node_col_scene1[i] = (0,0,0,0)  # greyish for non selected nodes
        node_pos_scene1[i] = generate_random_spherical_position()


In [34]:
# get all links between nucleoli nodes

nodelist_nucleoli_IDonly = [n.split("_",1)[1] for n in nodelist_nucleoli]

print("number of colored / visible links in scene1: ", len(linklist))

number of colored / visible links in scene1:  6994


In [35]:

link_col_scene1 = {}
linklist = []
for i, (u, v) in enumerate(G.edges()):
    if u in nodelist_nucleoli or v in nodelist_nucleoli or u in nodelist_nucleoli_IDonly or v in nodelist_nucleoli_IDonly:
        link_col_scene1[i] = G.nodes[u]['nodecolor']  # use node color for link color
        linklist.append((u, v))
    else:
        link_col_scene1[i] = (0, 0, 0, 0)  # transparent for non nucleoli links

# adapt link_col_scene1 to new link IDs
link_col_scene1_mapped = {
    link_mapping[(u, v)]: link_col_scene1[idx]
    for idx, (u, v) in enumerate(G.edges())
}

In [25]:
layout_name = "01_nucleolus"
new_tex_nodes = tex_gen.generate_node_color_texture(node_col_scene1, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(link_col_scene1_mapped, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(node_pos_scene1, layout_name, save=True, normalize_flag=False)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


### scene 2 - nucleoli + other compartments

In [36]:
# get all nodes with attrlist "compartment" "nucleoli"
compartments_to_consider = ['nucleoli', 'nuclear membrane', 'mitochondria', 'golgi apparatus']

nodelist_scene2 = []
for n, attr in G.nodes(data=True):
    if 'annotation' in attr and 'compartment' in attr['annotation']:
        if attr['annotation']['compartment'] in compartments_to_consider:
            nodelist_scene2.append(n)

In [37]:
# prepare vis parameters like node colors, link colors
import random 

node_col_scene2 = {}
node_pos_scene2 = {}

for i,n in enumerate(G.nodes()):
    if n in nodelist_scene2:
        #print("matching node: ", n)
        node_col_scene2[i] = G.nodes[n]['nodecolor']
        node_pos_scene2[i] = G.nodes[n]['pos']
    elif n not in nodelist_scene2: 
        node_col_scene2[i] = (0,0,0,0)  # greyish for non selected nodes
        node_pos_scene2[i] = generate_random_spherical_position()

In [38]:
link_col_scene2 = {}
linklist = []
for i, (u, v) in enumerate(G.edges()):
    if u in nodelist_scene2 and v in nodelist_scene2:
        link_col_scene2[i] = G.nodes[u]['nodecolor']  # use node color for link color
        linklist.append((u, v))
    else:
        link_col_scene2[i] = (0,0,0,0)  # transparent for non nucleoli links

# adapt link_col_scene1 to new link IDs
link_col_scene2_mapped = {
    link_mapping[(u, v)]: link_col_scene2[idx]
    for idx, (u, v) in enumerate(G.edges())
}
print("number of colored / visible links in scene2: ", len(linklist))

number of colored / visible links in scene2:  14418


In [39]:
layout_name = "02_nucleolu_andothers"
new_tex_nodes = tex_gen.generate_node_color_texture(node_col_scene2, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(link_col_scene2_mapped, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(node_pos_scene2, layout_name, save=True, normalize_flag=False)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


✅ Connected to /main


### scene 3 - whole cell 

In [40]:
# prepare vis parameters like node colors, link colors
import random 

node_col_scene_cell = {}
node_pos_scene_cell = {}
for i,n in enumerate(G.nodes()):
    node_col_scene_cell[i] = G.nodes[n]['nodecolor']
    node_pos_scene_cell[i] = G.nodes[n]['pos']

In [41]:
# get all links between nucleoli nodes
link_col_scene_cell_mapped = link_col_scene2_mapped.copy()

In [42]:
layout_name = "03_SpatialPPI"
new_tex_nodes = tex_gen.generate_node_color_texture(node_col_scene_cell, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(link_col_scene_cell_mapped, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(node_pos_scene_cell, layout_name, save=True, normalize_flag=False)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


✅ Connected to /main
✅ Connected to /main
✅ Connected to /main


### scene 4 - communities

In [33]:
def process_compartment(df_compartment, compartment_name, node_lookup):
    """
    Process a single compartment to assign community colors to nodes.

    Args:
        df_compartment (pd.DataFrame): DataFrame containing compartment data.
        compartment_name (str): Name of the compartment.
        node_lookup (dict): Lookup dictionary mapping unique IDs to uniprot IDs.

    Returns:
        dict: A dictionary mapping node IDs to their community colors.
    """
    # Get unique communities for the compartment
    unique_communities = df_compartment['community_id'].unique()
    num_communities = len(unique_communities)
    #print(f"Processing compartment: {compartment_name}, Number of communities: {num_communities}, Unique communities: {unique_communities}")
    
    # Create colormap and assign colors to communities
    cmap = plt.cm.get_cmap('rainbow', num_communities)
    community_color_map = {
        comm: (60, 60, 60, 60) if comm == -1 else tuple(int(c * 255) for c in cmap(i)[:3]) + (120,)
        for i, comm in enumerate(unique_communities)
    }
    
    # Assign colors to nodes based on community_id
    node_colors = {}
    for unique_id, uniprot_id in node_lookup.items():
        if compartment_name in unique_id:
            community_id = df_compartment.loc[df_compartment['Node ID'] == uniprot_id, 'community_id']
            if not community_id.empty:
                node_colors[unique_id] = community_color_map.get(community_id.values[0], (10, 10, 10, 10))
            else:
                node_colors[unique_id] = (10, 10, 10, 10)
    
    return node_colors

In [34]:
node_colors_communities_nucleolus = process_compartment(df_nucleoli, 'nucleoli', node_lookup)
print("number of nodes with community colors in nucleoli: ", len(node_colors_communities_nucleolus))

node_colors_community_actin = process_compartment(df_actin, 'actin', node_lookup)
print("number of nodes with community colors in actin: ", len(node_colors_community_actin))

node_colors_community_centro = process_compartment(df_centro, 'centro', node_lookup)
print("number of nodes with community colors in centrosome: ", len(node_colors_community_centro))

node_colors_community_cytosol = process_compartment(df_cytosol, 'cytosol', node_lookup)
print("number of nodes with community colors in cytosol: ", len(node_colors_community_cytosol))

node_colors_community_endoplas = process_compartment(df_endoplas, 'endoplas', node_lookup)
print("number of nodes with community colors in endoplasmic reticulum: ", len(node_colors_community_endoplas))

node_colors_community_golgi = process_compartment(df_golgi, 'golgi', node_lookup)
print("number of nodes with community colors in golgi apparatus: ", len(node_colors_community_golgi))

node_colors_community_intfil = process_compartment(df_intfil, 'intfil', node_lookup)
print("number of nodes with community colors in intermediate filaments: ", len(node_colors_community_intfil))

node_colors_community_microtub = process_compartment(df_microtub, 'microtub', node_lookup)
print("number of nodes with community colors in microtubules: ", len(node_colors_community_microtub))

node_colors_community_mito = process_compartment(df_mito, 'mito', node_lookup)
print("number of nodes with community colors in mitochondria: ", len(node_colors_community_mito))

node_colors_community_nuclearmem = process_compartment(df_nuclearmem, 'nuclearmem', node_lookup)
print("number of nodes with community colors in nuclear membrane: ", len(node_colors_community_nuclearmem))

node_colors_community_nuclplasm = process_compartment(df_nuclplasm, 'nuclplasm', node_lookup)
print("number of nodes with community colors in nucleoplasm: ", len(node_colors_community_nuclplasm))

node_colors_community_plasmmem = process_compartment(df_plasmmem, 'plasmmem', node_lookup)
print("number of nodes with community colors in plasma membrane: ", len(node_colors_community_plasmmem))

node_colors_community_prim = process_compartment(df_prim, 'prim', node_lookup)
print("number of nodes with community colors in primary cilium: ", len(node_colors_community_prim))

C:\Users\chris\AppData\Local\Temp\ipykernel_22036\2661854967.py:19: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = plt.cm.get_cmap('rainbow', num_communities)


number of nodes with community colors in nucleoli:  881
number of nodes with community colors in actin:  492
number of nodes with community colors in centrosome:  493
number of nodes with community colors in cytosol:  5738
number of nodes with community colors in endoplasmic reticulum:  1146
number of nodes with community colors in golgi apparatus:  1009
number of nodes with community colors in intermediate filaments:  54
number of nodes with community colors in microtubules:  576
number of nodes with community colors in mitochondria:  364
number of nodes with community colors in nuclear membrane:  88
number of nodes with community colors in nucleoplasm:  5555
number of nodes with community colors in plasma membrane:  3535
number of nodes with community colors in primary cilium:  393


In [35]:
# merge all dicts into one color dict 

# IMPORTANT: This order needs to match the Graph order: 
# [G_actin, G_centro, G_cytosol, G_endoplas, G_golgi, G_intfil, G_microtub, G_mito, G_nuclearmem, G_nucleoli, G_nuclplasm, G_plasmmem, G_prim]

node_colors_communities_all = {}
node_colors_communities_all.update(node_colors_community_actin)
node_colors_communities_all.update(node_colors_community_centro)
node_colors_communities_all.update(node_colors_community_cytosol)
node_colors_communities_all.update(node_colors_community_endoplas)
node_colors_communities_all.update(node_colors_community_golgi)
node_colors_communities_all.update(node_colors_community_intfil)
node_colors_communities_all.update(node_colors_community_microtub)
node_colors_communities_all.update(node_colors_community_mito)
node_colors_communities_all.update(node_colors_community_nuclearmem)
node_colors_communities_all.update(node_colors_communities_nucleolus)
node_colors_communities_all.update(node_colors_community_nuclplasm)
node_colors_communities_all.update(node_colors_community_plasmmem)
node_colors_communities_all.update(node_colors_community_prim)
print("total number of nodes with community colors in all compartments: ", len(node_colors_communities_all))
print("number of nodes in total graph: ", len(G.nodes()))

total number of nodes with community colors in all compartments:  20324
number of nodes in total graph:  20324


In [40]:
node_colors_communities_all_final = {}
for x,n in enumerate(G.nodes()):
    if n in node_colors_communities_all:
        node_colors_communities_all_final[x] = node_colors_communities_all[n]

In [41]:
layout_name = "04_communities"
new_tex_nodes = tex_gen.generate_node_color_texture(node_colors_communities_all_final, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(link_col_scene2_mapped, layout_name, save=True) # link_colors_community_nucleoli_mapped
new_tex_nodepos = tex_gen.generate_node_position_texture(node_pos_scene_cell, layout_name, save=True, normalize_flag=False)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


✅ Connected to /main
✅ Connected to /main
✅ Connected to /main


# ------- 

In [ ]:
# get hub nodes in the network with more than n connections
nn_connections = 300 # = 10hubs
hub_nodes = [n for n, d in G.degree() if d >= nn_connections]
print("number of hubnodes: ", len(hub_nodes))

number of hubnodes:  29


In [ ]:
node_col_hubnodes = {}
for i,n in enumerate(G.nodes()):
    if n in hub_nodes:
        node_col_hubnodes[i] = (255,0,0,255) #G.nodes[n]['nodecolor']
        #node_pos_hubnodes[i] = G.nodes[n]['pos']
    else:
        #node_pos_hubnodes[i] = generate_random_spherical_position((0.99,1.0))
        node_col_hubnodes[i] = (50,50,50,50)  # Set alpha to 255 for hub nodes# greyish for non selected nodes

link_colors_non = {}
for i, (u, v) in enumerate(G.edges()):
    link_colors_non[i] = (0,0,0,0)  # transparent for non nucleoli links

# ----------------------------------------------------------------------------------------------
# ONLY APPLES IN CASE OF INLCUDED INTER-Connections (see above when Graph G is constructed)

# DOES NOT WORK DUE TO MAPPING ISSUE (node iD including compartment prefix vs original ID)
#link_col_hubnodes = {}
#for i, (u, v) in enumerate(G.edges()):
#    if u in hub_nodes or v in hub_nodes:
#        link_col_hubnodes[i] = G.nodes[u]['nodecolor']  # use node color for link color
#    else:
#        link_col_hubnodes[i] = (0,0,0,0)  # transparent for non hub links

#link_col_hubnodes_mapped = {
#    link_mapping[(u, v)]: link_col_hubnodes[idx]    
#    for idx, (u, v) in enumerate(G.edges())
#}

In [ ]:
layout_name = "05_hubs"
new_tex_nodes = tex_gen.generate_node_color_texture(node_col_hubnodes, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(link_colors_non, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(node_pos_wholecell, layout_name, save=True, normalize_flag=False)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


In [ ]:
# List of all compartments to consider and their corresponding dataframes
compartments = {
    'actin': df_actin,
    'centrosome': df_centro,
    'cytosol': df_cytosol,
    'endoplasmic_reticulum': df_endoplas,
    'golgi': df_golgi,
    'intermediate_filaments': df_intfil,
    'microtubules': df_microtub,
    'mitochondria': df_mito,
    'nuclear_membrane': df_nuclearmem,
    'nucleoli': df_nucleoli,
    'nucleoplasm': df_nuclplasm,
    'plasma_membrane': df_plasmmem,
    'primary_cilium': df_prim
}

# Dictionary to store node colors for all compartments
node_colors_community_all = {}

for compartment, df_compartment in compartments.items():
    # Get unique communities for the compartment
    unique_communities = df_compartment['community_id'].unique()
    num_communities = len(unique_communities)
    print(f"Compartment: {compartment}, Number of communities: {num_communities}, Unique communities: {unique_communities}")
    
    # Create colormap and assign colors to communities
    cmap = plt.cm.get_cmap('rainbow', num_communities)
    community_color_map = {
        comm: (60, 60, 60, 60) if comm == -1 else tuple(int(c * 255) for c in cmap(i)[:3]) + (120,)
        for i, comm in enumerate(unique_communities)
    }
    
    # Assign colors to nodes based on community_id
    for unique_id, uniprot_id in node_lookup.items():
        if compartment in unique_id:
            community_id = df_compartment.loc[df_compartment['Node ID'] == uniprot_id, 'community_id']
            if not community_id.empty:
                node_colors_community_all[unique_id] = community_color_map.get(community_id.values[0], (10, 10, 10, 10))
            else:
                node_colors_community_all[unique_id] = (10, 10, 10, 10)

# Map node colors to graph-sorted order
node_colors_community_all_graphsorted = {
    x: node_colors_community_all.get(n, (10, 10, 10, 10)) for x, n in enumerate(G.nodes())
}

Compartment: actin, Number of communities: 8, Unique communities: [6 0 3 1 4 2 7 5]


C:\Users\chris\AppData\Local\Temp\ipykernel_18924\1198102930.py:28: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = plt.cm.get_cmap('rainbow', num_communities)


Compartment: centrosome, Number of communities: 13, Unique communities: [ 4  1  9  6  7  3 11  0  2  8 10  5 -1]
Compartment: cytosol, Number of communities: 16, Unique communities: [ 3 14 12  1  2  7  0 11 10  4  8  9 -1 13  6  5]
Compartment: endoplasmic_reticulum, Number of communities: 13, Unique communities: [ 2  4  7  3  1  9  6 10  0  5  8 -1 11]
Compartment: golgi, Number of communities: 16, Unique communities: [ 0  5 11 10  1  9  4 12 14 13  7  6  3  8  2 -1]
Compartment: intermediate_filaments, Number of communities: 6, Unique communities: [-1  1  0 11  2  8]
Compartment: microtubules, Number of communities: 13, Unique communities: [ 7  8 10  3  1  0  5  9 -1 11  4  6  2]
Compartment: mitochondria, Number of communities: 8, Unique communities: [ 2  1  8  0  5  7 10  6]
Compartment: nuclear_membrane, Number of communities: 12, Unique communities: [ 2  6 10  8  1 -1  4  7  0  9  3 14]
Compartment: nucleoli, Number of communities: 12, Unique communities: [ 0  9  4  2  3  7  5  8

In [ ]:
layout_name = "07_communities_more"
new_tex_nodes = tex_gen.generate_node_color_texture(node_colors_community_all_graphsorted, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(link_colors_non, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(node_pos_wholecell, layout_name, save=True, normalize_flag=False)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)
